# 🎭 Avatar Chok — ta photo qui parle (gratuit, Kaggle)

**Avant de lancer :**
1. À droite, **Session options → Accelerator : GPU T4 x2** (ou P100) et **Internet : On**.
2. **Add Input → Upload** : envoie **ta photo**, et au choix :
   - un fichier nommé **echantillon** (ta voix qui parle 30 s à 1 min) → l'avatar dira le TEXTE **avec ta voix clonée** ;
   - ou un enregistrement normal → l'avatar dit exactement cet enregistrement ;
   - ou rien → voix IA « Rémy ».
3. Modifie les réglages de la cellule ① si besoin, puis **Run All** (▶▶).

À la fin, la vidéo `avatar_chok.mp4` apparaît en bas et dans **Output**, à télécharger.

In [ ]:
# ① RÉGLAGES — modifie ici
TEXTE = """Salut, moi c'est Chok, je viens de Vénissieux. Bienvenue chez moi ! Ici, je vous montre comment l'intelligence artificielle peut vous faire gagner du temps : des sites pour vos commerces, des vidéos, des bons plans, et même de la musique. Vous avez un projet, une idée, un commerce ? Dites-moi en commentaire comment je peux vous aider. Abonnez-vous, et on avance ensemble, tranquille, posé."""
CLONER_MA_VOIX = True  # True : si tu as mis un fichier « echantillon » (ta voix 30 s à 1 min), l'avatar dit TEXTE avec TA voix
VOIX = "fr-FR-RemyMultilingualNeural"   # autres : fr-FR-HenriNeural, fr-FR-VivienneMultilingualNeural
CADRAGE = "full"      # "full" = toute la photo (chaise, bureau) · "crop" = visage seul
TETE_IMMOBILE = False # True si la tête « flotte » bizarrement en mode full
EXPRESSION = 1.2      # 1.0 à 1.5 : intensité des expressions


In [ ]:
# ② Vérifie la carte graphique
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ③ Installation (3 à 5 min)
%cd /kaggle/working
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /kaggle/working/SadTalker
!pip install -q kornia==0.6.8 face_alignment==1.3.5 yacs pydub gfpgan basicsr facexlib safetensors resampy imageio-ffmpeg edge-tts av 2>&1 | tail -n 2
# Correctifs de compatibilité (versions récentes de Python / numpy / torchvision / librosa)
import re, glob, site, os
for p in glob.glob('/kaggle/working/SadTalker/src/**/*.py', recursive=True):
    s = open(p).read(); n = re.sub(r'np\.float(?![0-9_])', 'np.float64', s); n = re.sub(r'np\.int(?![0-9_a-z])', 'int', n)
    n = n.replace('librosa.filters.mel(hp.sample_rate, hp.n_fft,', 'librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,')
    if n != s: open(p, 'w').write(n)
for sp in site.getsitepackages():
    f = os.path.join(sp, 'basicsr/data/degradations.py')
    if os.path.exists(f):
        s = open(f).read().replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional'); open(f, 'w').write(s)
print('OK installation')

In [ ]:
# ④ Modèles (environ 2 Go)
!bash scripts/download_models.sh 2>&1 | tail -n 3
!ls checkpoints gfpgan/weights 2>/dev/null | head

In [ ]:
# ⑤ Ta photo et ta voix
import glob, os, subprocess
files = glob.glob('/kaggle/input/**/*', recursive=True)
photos = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))]
audios = [f for f in files if f.lower().endswith(('.mp3','.wav','.m4a','.ogg','.aac','.webm','.opus'))]
echant = [f for f in audios if 'echantillon' in os.path.basename(f).lower()]
autres = [f for f in audios if f not in echant]
assert photos, "❌ Aucune photo trouvée : Add Input → Upload ta photo."
PHOTO = photos[0]; print('📸 Photo :', PHOTO)
def to_wav(src, dst, sr=16000):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',src,'-ac','1','-ar',str(sr),dst], check=True); return dst
if echant and CLONER_MA_VOIX:
    print('🧬 Clonage de ta voix à partir de :', echant[0])
    !pip install -q coqui-tts 2>&1 | tail -n 1
    os.environ['COQUI_TOS_AGREED'] = '1'
    ref = to_wav(echant[0], '/kaggle/working/echantillon.wav', 22050)
    from TTS.api import TTS
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
    VOIX_FICHIER = '/kaggle/working/voix.wav'
    tts.tts_to_file(text=TEXTE, speaker_wav=ref, language='fr', file_path=VOIX_FICHIER)
    print('✅ Texte dit avec ta voix clonée')
elif autres:
    VOIX_FICHIER = to_wav(autres[0], '/kaggle/working/voix.wav'); print('🎙️ Ton enregistrement :', autres[0])
else:
    VOIX_FICHIER = '/kaggle/working/voix.mp3'
    !edge-tts --voice "$VOIX" --text "$TEXTE" --write-media "$VOIX_FICHIER"
    print('🗣️ Voix IA :', VOIX)
from IPython.display import Audio, display; display(Audio(VOIX_FICHIER))

In [ ]:
# ⑥ Création de l'avatar qui parle (1 à 5 min selon la longueur)
cmd = f'python inference.py --driven_audio "{VOIX_FICHIER}" --source_image "{PHOTO}" --result_dir /kaggle/working/resultats --preprocess {CADRAGE} --enhancer gfpgan --expression_scale {EXPRESSION} --size 512'
if TETE_IMMOBILE: cmd += ' --still'
print(cmd); !{cmd}

In [ ]:
# ⑦ Résultat
import glob, shutil
from IPython.display import Video, display
res = sorted(glob.glob('/kaggle/working/resultats/**/*.mp4', recursive=True), key=os.path.getmtime)
assert res, "❌ Pas de vidéo : envoie une capture de l'erreur à Claude."
shutil.copy(res[-1], '/kaggle/working/avatar_chok.mp4')
print('✅ Vidéo prête : avatar_chok.mp4 (onglet Output à droite)')
display(Video('/kaggle/working/avatar_chok.mp4', embed=True, width=360))